### Indhold - TLDR
I disse opgaver skal i arbejde med:
- Torch autograd kan differentiere for os
- Rigtig mange datapunkter og SGD
- Smartere gradient descent-skridt
  - normal gradient descent
  - momentum
  - RMSprop
  - Adam
  - ekstra opgave: design jeres egen variant

---

### Introduktion

I regression 1 fandt I selv ud af hvor meget I skulle flytte jeres parametre for at forbedre modellen — ved at differentiere loss-funktionen i hånden, og tage ét lille skridt ad gangen ned ad loss-fladen.

Det virkede fint for de simple modeller I byggede selv. Men forestil jer en model med tusindvis af parametre og en helt igennem kompliceret loss-funktion — der bliver det upraktisk (og nogle gange umuligt) at differentiere i hånden. Det er præcis det problem autograd løser: PyTorch følger med i alle udregninger I laver, og kan bagefter regne den afledte ud for jer, uanset hvor kompliceret udregningen er.

Med gradienten på plads er der stadig spørgsmålet om hvordan man bedst bruger den til at tage skridt. Almindelig gradient descent er ikke altid den hurtigste vej ned — der findes smartere metoder, som I selv skal bygge og sammenligne i denne blok.

### 1: Gradienten med autograd

I regression 1 lærte I at differentiere med potensreglen, sumreglen og kædereglen (opgave 5), og brugte det til selv at udlede gradienten af loss i hånden (opgave 7-8).

PyTorch kan gøre nøjagtig det samme automatisk. Gør en variabel til en **tensor** med `requires_grad=True`, byg en udregning med den, og kald `.backward()` — så regner PyTorch den afledte ud, og gemmer den i `.grad`. Det virker på lige så komplicerede udregninger som dem I selv differentierede i opgave 5:

In [ ]:
# Henter hjælpefunktioner + testfil fra GitHub (Plan B: upload dem manuelt via mappeikonet i Colab)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/98-Helpers/helpers.py
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/02-Regression/regression2/regression2_test.py

import numpy as np
import helpers as sesy_viz          # sesy_vizualisation er foldet ind i helpers.py
from regression2_test import *

In [ ]:
import torch

# husk opgave 5.1: f(x) = 5x²  →  f'(x) = 10x  (potensreglen)
def f(x):
    return 5 * x**2

x = torch.tensor(3.0, requires_grad=True)
y = f(x)
y.backward()
print(f"f(3.0)  = {y.item()}")        # facit: 5*3² = 45
print(f"f'(3.0) = {x.grad.item()}")   # facit: 10*3 = 30 — det autograd fandt i .grad

In [ ]:
# husk opgave 5.7: f(x) = (2x+1)²  →  f'(x) = 4(2x+1)  (kædereglen)
def g(x):
    return (2*x + 1)**2

x = torch.tensor(2.0, requires_grad=True)
y = g(x)
y.backward()
print(f"g(2.0)  = {y.item()}")        # facit: (2*2+1)² = 25
print(f"g'(2.0) = {x.grad.item()}")   # facit: 4*(2*2+1) = 20 — det autograd fandt i .grad

In [ ]:

# husk opgave 7.2 + 8.1: MSE for linjen y=a*x+b ift. 2 punkter — nu med gradienten mht. BÅDE a og b på én gang
def mse(a, b, punkter):
    n = len(punkter)
    total = 0
    for x, y in punkter:
        total += (a*x + b - y)**2
    return total / n

punkter = [(1, 2), (2, 4)]
a = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
y = mse(a, b, punkter)
y.backward()
print(f"mse(0,0) = {y.item()}")                            # facit: ((0-2)² + (0-4)²)/2 = 10
print(f"d/da = {a.grad.item()}, d/db = {b.grad.item()}")   # facit: jeres eget opg7.2/8.1-facit

### Opgave 1)

I skal nu reflektere kort over hvad I lige har set autograd gøre.

**Opgave 1.1)**
Med egne ord: hvad gør `requires_grad=True`, og hvad ligger der i `.grad` efter I har kaldt `.backward()`? Hvordan svarer det til det I selv gjorde i hånden i opgave 5-8?

In [ ]:
# beskriv med jeres egne ord hvad requires_grad, .backward() og .grad gør
opg1_1 = """
...
"""
test_opg1_1(opg1_1)

Nu bruger vi det samme værktøj på selve loss-funktionen fra regression 1 (MSE for linjen $y=a\cdot x+b$) — bare med 100 punkter i stedet for 4:

In [ ]:
import torch

punkter = sesy_viz.LINJE_PUNKTER   # ~100 punkter, normalfordelt omkring en linje

# jf. regression1 opgave 3+4: MSE for linjen y=a*x+b (samme loss som i regression 1)
def loss(a, b, punkter):
    n = len(punkter)
    total = 0
    for x, y in punkter:
        total += (a*x + b - y)**2
    return total / n

# jf. regression1 opgave 8: gradienten af loss - nu med autograd i stedet for i hånden:
# gør a og b til tensorer, kald loss(a, b).backward(), og aflæs .grad
def gradient(a, b, punkter):
    a = torch.tensor(float(a), dtype=torch.float64, requires_grad=True)
    b = torch.tensor(float(b), dtype=torch.float64, requires_grad=True)
    loss(a, b, punkter).backward()
    return a.grad.item(), b.grad.item()

# jf. regression1 opgave 9: ét gradient descent-skridt
def step(a, b, punkter, lr):
    da, db = gradient(a, b, punkter)
    return a - lr*da, b - lr*db

**Opgave 1.2)**
I cellen ovenfor bruges `gradient(a, b, punkter)` (som selv bruger `.backward()`) inde i `step(a, b, punkter, lr)`, for hvert eneste skridt ned ad loss-fladen. Beskriv med egne ord hvilken rolle autograd spiller i denne løkke, sammenlignet med opgave 9 i regression 1 — hvor I selv udledte og indsatte gradientformlen.

In [ ]:
# beskriv med jeres egne ord hvilken rolle autograd spiller i træningsløkken ovenfor
opg1_2 = """
...
"""
test_opg1_2(opg1_2)

Her er en illustration af losslandskabet og af, hvordan gradient descent træner med alle 100 punkter:


In [ ]:

a, b = 0.0, 0.0
lr = 0.02

path = [(a, b)]
paneler = []
for _ in range(50):
    a, b = path[-1]
    grad_a, grad_b = gradient(a, b, punkter)
    paneler.append((
        sesy_viz.modelfit(a, b, punkter, x_range=(0,10), y_range=(-2,14)),
        sesy_viz.loss_kontur(lambda a, b: loss(a, b, punkter), a_range=(-1,3), b_range=(-3,3), resolution=40, path=list(path), gradient=(grad_a, grad_b), colorbar=False),
        sesy_viz.loss_over_tid([loss(a, b, punkter) for a, b in path]),
    ))
    path.append(step(a, b, punkter, lr))

print("Beregning færdig, samler til animation:")
sesy_viz.animate(paneler, max_frames=25)

### 2: Minibatch og stokastisk gradient descent

Hvorfor regne gradienten ud fra alle punkter for hvert skridt, hvis vi kan nøjes med et tilfældigt udpluk?

Med 100 punkter (og for rigtige datasæt ofte millioner) er det dyrt at bruge alle punkter hvert skridt. Løsningen er at trække et tilfældigt udpluk — en **batch** — og regne gradienten ud fra den. Den gradient peger næsten samme vej som gradienten fra alle punkter, men er meget billigere. Det kaldes **minibatch** eller **stokastisk gradient descent (SGD)**.

`random.sample(punkter, k)` tager en liste som input, vælger tilfældigt k af dem, og giver det som output — I kender ikke `random`-biblioteket endnu, men den er let at bruge.

### Opgave 2)

**Opgave 2.1)**
Skriv træningsløkken `opg2_1`: træk en ny batch med `random.sample`, og tag ét `skridt` (fra afsnit 1) — `n` gange i træk.

In [ ]:
import random

# random.sample(punkter, k) tager en liste som input, vælger tilfældigt k af dem, og giver det som output
# træningsloop: træk en ny batch med random.sample, og tag ét skridt (skridt, fra recap), n gange i træk
def opg2_1(a, b, punkter, batch_size, lr, n):
    ...
    return a, b
test_opg2_1(opg2_1)

Illustrationen viser, hvordan loss-konturen ændrer sig med forskellige batch-størrelser, og hvordan SGD-stien ser ud:


In [ ]:
batch_size = 5   # prøv fx 1 (meget jitter) eller 50 (næsten roligt) og kør cellen igen

full_loss_fn = lambda a, b: loss(a, b, punkter)   # landskabet regnet ud fra alle punkter — stabilt facit-landskab

a, b = 0.0, 0.0
lr = 0.02

path = [(a, b)]
paneler = []
for _ in range(100):
    a, b = path[-1]
    batch_idx = random.sample(range(len(punkter)), batch_size)   # samme udtræk som i opg2_1, men som indices så vi kan fremhæve dem
    batch = [punkter[i] for i in batch_idx]
    grad_a, grad_b = gradient(a, b, batch)                # kun til at tegne pilen — selve skridtet tages nedenfor
    batch_loss_fn = lambda a, b, batch=batch: loss(a, b, batch)   # landskabet regnet kun ud fra denne batch
    paneler.append((
        sesy_viz.modelfit(a, b, punkter, x_range=(0,10), y_range=(-2,14),
                           highlight=batch_idx, highlight_label='i batch', rest_label='resten'),
        sesy_viz.loss_kontur(full_loss_fn, a_range=(-1,3), b_range=(-3,3), resolution=40, path=list(path), gradient=(grad_a, grad_b), colorbar=False, title='loss(alle punkter)'),
        sesy_viz.loss_kontur(batch_loss_fn, a_range=(-1,3), b_range=(-3,3), resolution=40, path=list(path), gradient=(grad_a, grad_b), colorbar=False, title='loss(kun denne batch)'),
    ))
    path.append(step(a, b, batch, lr))   # skridt fra recap, på samme batch

print("Beregning færdig, samler til animation:")
sesy_viz.animate(paneler, max_frames=50)

**Opgave 2.2)**
Kør cellen ovenfor med forskellige `batch_size` (1, 5, 20, 50, 100). Beskriv med egne ord hvad der ændrer sig:

In [ ]:
# beskriv med jeres egne ord hvad der sker ved forskellige batch-størrelser
opg2_2 = """
...
"""
test_opg2_2(opg2_2)

### 3: Smartere optimizers

Kan vi komme hurtigere/mere direkte til minimum, uden nødvendigvis at bruge flere punkter pr. skridt? Det er det bl.a. Adam — en af de mest populære optimizers — prøver at gøre.

I skal bygge fire metoder (gd, momentum, rmsprop, adam), hver i to lag: selve opdateringen (testet for sig) og løkken udenom (testet som den fulde metode) — alle med samme form `(loss, start, ...)`. I får kun `loss`; gradienten finder I selv med `.backward()` (se `get_gradient` nedenfor).

**Constraints:** kørslen stopper automatisk (og bruger jeres seneste punkt) hvis `loss` kaldes mere end **100** gange, eller hvis I ender mere end **1000** væk fra landskabet — det sidste er nok en fejl, ikke en strategi.

Testes på 4 landskaber (linjefitting + 3 ukendte), hver fra 4 udvalgte startpunkter. Brug `sesy_viz.evaluer_gd_metode([("gd", opg3_2), ...])` til at sammenligne jeres metoder undervejs.

In [ ]:
# I får kun loss(a, b) - ikke gradienten. Men den finder I selv med autograd:
# gør a og b til tensorer, kald loss(a, b).backward(), og aflæs .grad. Samme .backward()-trick
# som i afsnit 1, nu for et hvilket som helst loss.
def get_gradient(loss, a, b):
    a = torch.tensor(float(a), dtype=torch.float64, requires_grad=True)
    b = torch.tensor(float(b), dtype=torch.float64, requires_grad=True)
    loss(a, b).backward()
    return a.grad.item(), b.grad.item()

### Almindelig gradient descent

$$(a,b)_{t+1} = (a,b)_t - \text{lr}\cdot \nabla L(a_t, b_t)$$

Samme `gradient`+`skridt` fra recap, bare opskrevet i formen `(loss, start)` som resten af blokken bruger.

### Opgave 3)

Byg gradient descent i to lag, testet hver for sig i cellen nedenfor. `n` styrer hvor mange skridt der tages — 100 som default, hvilket bruger hele budgettet på 100 loss-kald.

**Opgave 3.1)**
Ét gradient descent-skridt: find gradienten af `loss` med `.backward()`, og træk `lr` gange gradienten fra.

**Opgave 3.2)**
Løkken udenom: kald jeres `opg3_1` `n` gange i træk.

In [ ]:
# ét gradient descent-skridt: find gradienten af loss med .backward(), og træk lr*gradienten fra
def opg3_1(a, b, loss, lr):
    da, db = get_gradient(loss, a, b)
    ...
    return a, b
test_opg3_1(opg3_1)

# løkken udenom: kald opg3_1 n gange i træk
def opg3_2(loss, start, lr=0.01, n=100):
    a, b = start
    ...
    return a, b
test_opg3_2(opg3_2)

sesy_viz.evaluer_gd_metode([("gd", opg3_2)])

**Ekstra Opgave 3.3)**

`opg3_2` klarer sig især dårligt på `rosenbrock`, (hvor hældningerne er ekstremt store).
Prøv at brainstorme hvordan man kunne sørge for at modellen ikke tager alt for store skridt.
Prøv fx en anden `lr`, at begrænse størelsen af skridtet, eller en helt anden idé.

In [ ]:
""" # fjern kommentering og skriv jeres egen variant her
def opg3_3(loss, start, lr=0.02, n=100):
    ...

sesy_viz.evaluer_gd_metode([
    ("gd", opg3_2),
    ("egen v1", opg3_3),
])
"""

### Momentum

Momentum er **ikke** en fysisk "bold der ruller ned ad bakken" — det er en **vægtet, aftagende
gennemsnit af alle de gradienter man har set indtil videre**:

*syntax note: subscript $t$ betyder "ved skridt $t$" — $v_{t-1}$ er værdien fra forrige skridt, $v_t$ er den vi regner lige nu*

$$v_t = \beta \cdot v_{t-1} + (1-\beta)\cdot \nabla L(a_t, b_t), \qquad (a,b)_{t+1} = (a,b)_t - \text{lr}\cdot v_t$$

$\beta$ (typisk omkring 0.9), er en hyperparameter som styrer hvor meget vi vægter gamle værdier, vs hvor meget vi vægter en ny opservation.
Gradient fra $k$ skridt tilbage tæller med vægten $(1-\beta)\beta^k$, altså mindre og mindre jo længere tilbage den er.

### Opgave 4)

Byg momentum i tre lag, ligesom `opg3_2` — bare med ét ekstra lag for selve $v$-formlen.

**Opgave 4.1)**
Selve $v$-formlen: opdater det glidende gennemsnit $v$ med den nye gradient vægtet ind.

**Opgave 4.2)**
Ét momentum-skridt, som bruger $v$ i stedet for selve gradienten.

**Opgave 4.3)**
Løkken udenom: kald jeres `opg4_2` `n` gange i træk, og hold styr på $v$ undervejs.

In [ ]:
# opdater det glidende gennemsnit v, med den nye gradient vægtet ind
def opg4_1(va, vb, da, db, beta):
    ...
    return va, vb
test_opg4_1(opg4_1)

# ét momentum-skridt: find gradienten, opdater v, brug så v (ikke selve gradienten) til skridtet
def opg4_2(a, b, va, vb, loss, lr, beta):
    da, db = get_gradient(loss, a, b)
    ...
    return a, b, va, vb
test_opg4_2(opg4_2)

# løkken udenom: kald opg4_2 n gange i træk, og hold styr på v undervejs
def opg4_3(loss, start, lr=0.02, beta=0.9, n=100):
    ...
    return a, b
test_opg4_3(opg4_3)

Se momentum illustreret med jeres eget `opg4_2` kaldt for hvert billede: 
* venstre panel er gradienten lige nu (+ alle tidligere målinger, gråt), 
* midten samler dem til $v$ (ældre vejer mindre),
* højre er det skridt der rent faktisk tages:

In [ ]:
landscape = [l for l in sesy_viz.LANDSKABER if l["name"] == "plateau (1D)"][0]
lr, beta = 0.1, 0.9
show_scale = 1.0   # ren visningsskala for pilene — juster hvis de er for små/store til at se

a, b = 1, 0.6
va, vb = 0.0, 0.0
references = []   # hver tidligere gradients bidrag (1-beta)*grad, DA den var ny
measurements = []     # (a, b, ref_a, ref_b) — hvor + hvad, til panel 1's grå pile
path = [(a, b)]

frames = []
for _ in range(100):
    da, db = get_gradient(landscape["loss"], a, b)   # kun til visning i panel 1 (selve skridtet tages nedenfor)
    reference = (show_scale * (1 - beta) * da, show_scale * (1 - beta) * db)
    references.append(reference)
    measurements.append((a, b, *reference))

    # ét par pr. tidligere gradient: (reference = værdien DA den kom) og (nu = dens aktuelle,
    # vægtede bidrag til v_t — reference skrumpet med beta**alder)
    n = len(references)
    pair = [(ref, (ref[0] * beta**alder, ref[1] * beta**alder))
           for alder, ref in zip(range(n - 1, -1, -1), references)]

    ny_a, ny_b, va, vb = opg4_2(a, b, va, vb, landscape["loss"], lr, beta)   # jeres egen funktion

    shared = dict(path=list(path), colorbar=False)
    frames.append((
        sesy_viz.loss_kontur(landscape["loss"], landscape["a_range"], landscape["b_range"], gradient=(show_scale * da, show_scale * db),
                              extra_arrows=[(ma, mb, mda, mdb, "gray", 0.5) for ma, mb, mda, mdb in measurements],
                              title="gradient nu (+ tidligere målinger)", **shared),
        sesy_viz.vektor_vifte(pair, sum_vector=(show_scale * va, show_scale * vb), title="tidligere gradienter: original vs. vægtet"),
        sesy_viz.loss_kontur(landscape["loss"], landscape["a_range"], landscape["b_range"], gradient=(show_scale * va, show_scale * vb),
                              gradient_color="orange", title="momentum-skridt", **shared),
    ))
    a, b = ny_a, ny_b
    path.append((a, b))

sesy_viz.animate(frames, max_frames=50)

In [ ]:
sesy_viz.evaluer_gd_metode([
    ("gd", opg3_2),
    ("momentum", opg4_3),
])

**Ekstra Opgave 4.4)**

Prøv igen at bygge jeres egen variant, i kan tage inspiration fra momentum eller finde på en anden cooked ide, eller bygge videre på en af de tidligere ekstra opgaver.

In [ ]:
""" # fjern kommentering og skriv jeres egen variant her
def opg4_4(loss, start, lr=0.02, n=100):
    ...

sesy_viz.evaluer_gd_metode([
    ("gd", opg3_2),
    ("momentum", opg4_3),
    ("egen v2", opg4_4),
])
"""

### RMSprop

RMSprops idé: skalér hver akse med sin egen (nyligt sete) gradient-størrelse, så gradienten i den skalerede akse ligger omkring $\pm 1$. Dermed tages der lige store skridt i alle retninger, uanset om den oprindelige akse var stejl eller flad:

$$s_t = \beta\cdot s_{t-1}+(1-\beta)\cdot \nabla L(a_t,b_t)^2, \qquad (a,b)_{t+1} = (a,b)_t - \text{lr}\cdot\frac{\nabla L(a_t,b_t)}{\sqrt{s_t}+\varepsilon}$$

($s$ regnes **elementvis** pr. parameter — $a$ og $b$ får hver deres egen skalering. $\varepsilon$
er blot et lille tal, der forhindrer division med 0.)

### Opgave 5)

Byg RMSprop i tre lag, samme opskrift som momentum.

**Opgave 5.1)**
Selve $s$-formlen: opdater det glidende gennemsnit af gradienten i anden.

**Opgave 5.2)**
Ét RMSprop-skridt: skalér gradienten med $\sqrt{s}$ før skridtet.

**Opgave 5.3)**
Løkken udenom: kald jeres `opg5_2` `n` gange i træk, og hold styr på $s$ undervejs.

In [ ]:
# opdater det glidende gennemsnit s af gradienten i anden
def opg5_1(sa, sb, da, db, beta):
    ...
    return sa, sb
test_opg5_1(opg5_1)

# ét rmsprop-skridt: find gradienten, opdater s, og skalér gradienten med sqrt(s) før skridtet
def opg5_2(a, b, sa, sb, loss, lr, beta, eps):
    ...
    return a, b, sa, sb
test_opg5_2(opg5_2)

# løkken udenom: kald opg5_2 n gange i træk, og hold styr på s undervejs
def opg5_3(loss, start, lr=0.02, beta=0.5, eps=1e-8, n=100):
    ...
    return a, b
test_opg5_3(opg5_3)

se RMSprop animeret med jeres egen `opg5_2` kaldt for hvert billede: venstre panel er den rå gradient, midten er det **samme** rum omskaleret med $\sqrt{s_t}$ (en aflang dal bliver rund), højre er skridtet der rent faktisk tages:

In [ ]:
landscape = [l for l in sesy_viz.LANDSKABER if l["name"] == "linjefitting"][0]
lr, beta, eps = 0.02, 0.1, 1e-8   # I har selv valgt disse (ikke opg5_3's defaults)
show_scale = 0.01   # visningsskala for den rå gradient-pil (den rmsprop-skalerede pil er allerede ca. i størrelsesorden 1)
color_range = sesy_viz.loss_range(landscape["loss"], landscape["a_range"], landscape["b_range"])   # samme farveskala i alle paneler/billeder
a, b = 0, 0
sa, sb = 0.0, 0.0
path = [(a, b)]

frames = []
for _ in range(100):
    da, db = get_gradient(landscape["loss"], a, b)   # kun til visning i panel 1
    ny_a, ny_b, sa, sb = opg5_2(a, b, sa, sb, landscape["loss"], lr, beta, eps)   # jeres egen funktion
    scale_a, scale_b = sa**0.5 + eps, sb**0.5 + eps
    show_a, show_b = da / scale_a, db / scale_b   # den skalerede retning — det panel 2/3 viser

    # panel 2's a-akse beholder landskabets egen, faste range hele vejen igennem (aldrig
    # zoomet ind/ud pr. billede) — så I kan se rummet selv strække/klemme sig (aksetallene
    # ændrer sig med skala_a: store tal tidligt, tæt på 1 midtvejs, små sent). b-aksens
    # bredde sættes til at matche a-aksens (samme antal enheder i det omskalerede rum — et
    # ægte kvadrat, ikke et skævt rektangel), men centreret om nuværende b (ikke om 0), så
    # det er dér kurven rent faktisk bøjer der vises.
    width = (landscape["a_range"][1] - landscape["a_range"][0]) * scale_a
    lokal_b_range = (b - width / (2 * scale_b), b + width / (2 * scale_b))

    shared = dict(path=list(path), colorbar=False, color_range=color_range)
    frames.append((
        sesy_viz.loss_kontur(landscape["loss"], landscape["a_range"], landscape["b_range"], gradient=(show_scale * da, show_scale * db), title="gradient nu", **shared),
        sesy_viz.loss_kontur(landscape["loss"], landscape["a_range"], lokal_b_range, scale=(scale_a, scale_b),
                              gradient=(show_scale * da, show_scale * db),
                              extra_arrows=[(a, b, show_a, show_b, "orange", 1.0)],
                              title="skaleret rum", **shared),
        sesy_viz.loss_kontur(landscape["loss"], landscape["a_range"], landscape["b_range"], gradient=(show_a, show_b),
                              gradient_color="orange", title="rmsprop-skridt", **shared),
    ))
    a, b = ny_a, ny_b
    path.append((a, b))

sesy_viz.animate(frames, max_frames=50)

In [ ]:
sesy_viz.evaluer_gd_metode([
    ("gd", opg3_2),
    ("momentum", opg4_3),
    ("rmsprop", opg5_3),
])

**Ekstra Opgave 5.4)**

Prøv igen at bygge jeres egen variant, i kan tage inspiration fra momentum, rmsprop eller finde på en anden cooked ide, eller bygge videre på en af de tidligere ekstra opgaver.

In [ ]:
""" # fjern kommentering og skriv jeres egen variant her
def opg5_4(loss, start, lr=0.02, n=100):
    ...

sesy_viz.evaluer_gd_metode([
    ("gd", opg3_2),
    ("momentum", opg4_3),
    ("rmsprop", opg5_3),
    ("egen v3", opg5_4),
])
"""

### Adam

Adam kombinerer de to metoder ovenfor: $v_t$ fra momentum og $s_t$ fra RMSprop.
I kan genbruge jeres `opg4_1` og `opg5_1` fra før.
plus en ny **bias-korrektion** af begge, vigtig i de første skridt, hvor $v_0=s_0=0$ ellers trækker
gennemsnittet kunstigt mod nul:

$$v_t = \beta_1 v_{t-1} + (1-\beta_1)\nabla L(a_t,b_t), \qquad s_t = \beta_2 s_{t-1} + (1-\beta_2)\nabla L(a_t,b_t)^2$$

$$\hat v_t = \frac{v_t}{1-(\beta_1)^t}, \qquad \hat s_t = \frac{s_t}{1-(\beta_2)^t}, \qquad (a,b)_{t+1} = (a,b)_t - \text{lr}\cdot\frac{\hat v_t}{\sqrt{\hat s_t}+\varepsilon}$$

### Opgave 6)

**Opgave 6.1)**
Ét Adam-skridt: samme $v$/$s$-opdatering som momentum/RMSprop (genbrugt!), plus bias-korrektion af begge. (i kan kalde `opg4_1` og `opg5_1`)

**Opgave 6.2)**
Løkken udenom: kald jeres `opg6_1` `n` gange i træk
`t` starter ved 1 (bruges i bias-korrektionen).

In [ ]:
# ét adam-skridt: samme v/s-opdatering som momentum/rmsprop (genbrugt!), plus bias-korrektion af begge
def opg6_1(a, b, va, vb, sa, sb, t, loss, lr, beta1, beta2, eps):
    ...
    return a, b, va, vb, sa, sb
test_opg6_1(opg6_1)

# løkken udenom: kald opg6_1 n gange i træk - t starter ved 1 (bruges i bias-korrektionen)
def opg6_2(loss, start, lr=0.2, beta1=0.9, beta2=0.99, eps=1e-8, n=100):
    ...
    return a, b
test_opg6_2(opg6_2)

# alt I har bygget i denne blok, samlet i ét sidste overblik:
sesy_viz.evaluer_gd_metode([
    ("gd", opg3_2),
    ("momentum", opg4_3),
    ("rmsprop", opg5_3),
    ("adam", opg6_2),
])

se adam animeret, med jeres egen `opg6_1` kaldt for hvert billede.
- panel 1 er den originale gradient
- panel 2 er $\hat v_t$ (momentum-viften, korrigeret), 
- panel 3 er samme $\hat v_t$ (orange) skaleret ned med $\sqrt{\hat s_t}$ til en ny, magenta pil - den pil er adam-skridtet i panel 4:

In [ ]:
landscape = [l for l in sesy_viz.LANDSKABER if l["name"] == "rosenbrock"][0]
lr, beta1, beta2, eps = 0.2, 0.9, 0.99, 1e-8   # højere lr og lavere beta2 end opg6_2's
                                                # defaults — kun her, så begge korrektioner ses
                                                # tydeligt inden for 80 billeder
show_scale = 0.2   # visningsskala for den rå gradient/v̂-pil (v̂/ŝ er allerede ca. størrelsesorden 1)
color_range = sesy_viz.loss_range(landscape["loss"], landscape["a_range"], landscape["b_range"])
a, b = -1.0, -0.5
va, vb = 0.0, 0.0
sa, sb = 0.0, 0.0
history = []   # (da, db, ref_a, ref_b) — reference frosset fra dengang gradienten blev målt
measurements = []   # (a, b, ref_a, ref_b), til panel 1's grå pile
path = [(a, b)]

frames = []
for t in range(1, 101):
    da, db = get_gradient(landscape["loss"], a, b)   # kun til visning i panel 1
    korr1 = 1 / (1 - beta1**t)
    reference = (korr1 * show_scale * (1 - beta1) * da, korr1 * show_scale * (1 - beta1) * db)
    history.append((da, db, *reference))
    measurements.append((a, b, *reference))

    ny_a, ny_b, va, vb, sa, sb = opg6_1(a, b, va, vb, sa, sb, t, landscape["loss"], lr, beta1, beta2, eps)   # jeres egen funktion
    vha, vhb = va / (1 - beta1**t), vb / (1 - beta1**t)
    scale_a, scale_b = (sa / (1 - beta2**t))**0.5 + eps, (sb / (1 - beta2**t))**0.5 + eps

    # samme vifte-konstruktion som momentum: reference er frosset (med sin egen korr1 fra
    # dengang) — kun "nu"-bidraget bruger dagens korr1
    n = len(history)
    pair = [((ref_a, ref_b), (korr1 * show_scale * (1 - beta1) * beta1**alder * hda, korr1 * show_scale * (1 - beta1) * beta1**alder * hdb))
           for alder, (hda, hdb, ref_a, ref_b) in zip(range(n - 1, -1, -1), history)]

    v_show_a, v_show_b = show_scale * vha, show_scale * vhb   # v̂ (orange) — samlet i panel 2, bæres videre til panel 3
    sv_show_a, sv_show_b = vha / scale_a, vhb / scale_b     # v̂/ŝ (magenta) — nyt i panel 3, bæres videre til panel 4

    # a-aksen beholder landskabets egen, faste range hele vejen (samme idé som RMSprop-
    # panelet ovenfor) — b-aksens bredde matcher a's (et ægte kvadrat), centreret om
    # nuværende b.
    width = (landscape["a_range"][1] - landscape["a_range"][0]) * scale_a
    lokal_b_range = (b - width / (2 * scale_b), b + width / (2 * scale_b))

    shared = dict(path=list(path), colorbar=False)
    frames.append((
        sesy_viz.loss_kontur(landscape["loss"], landscape["a_range"], landscape["b_range"], gradient=(show_scale * da, show_scale * db),
                              extra_arrows=[(ma, mb, mda, mdb, "gray", 0.5) for ma, mb, mda, mdb in measurements],
                              title="gradient nu (+ tidligere målinger)", color_range=color_range, **shared),
        sesy_viz.vektor_vifte(pair, sum_vector=(v_show_a, v_show_b), title="v (korrigeret): momentum-vifte"),
        sesy_viz.loss_kontur(landscape["loss"], landscape["a_range"], lokal_b_range, scale=(scale_a, scale_b),
                              gradient=(v_show_a, v_show_b), gradient_color="orange",
                              extra_arrows=[(a, b, sv_show_a, sv_show_b, "magenta", 1.0)],
                              title="s (korrigeret): skaleret rum", color_range=color_range, **shared),
        sesy_viz.loss_kontur(landscape["loss"], landscape["a_range"], landscape["b_range"],
                              gradient=(sv_show_a, sv_show_b), gradient_color="magenta", title="adam-skridt", color_range=color_range, **shared),
    ))
    a, b = ny_a, ny_b
    path.append((a, b))

sesy_viz.animate(frames, max_frames=50)

**Ekstra Opgave 6.3)**

Prøv igen at bygge jeres egen variant, i kan tage inspiration fra momentum, rmsprop, adam eller finde på en anden cooked ide, eller bygge videre på en af de tidligere ekstra opgaver.

In [ ]:
""" # fjern kommentering og skriv jeres egen variant her
def opg6_3(loss, start, lr=0.01, n=100):
    ...

sesy_viz.evaluer_gd_metode([
    ("gd", opg3_2),
    ("momentum", opg4_3),
    ("rmsprop", opg5_3),
    ("adam", opg6_2),
    ("egen", opg6_3),
])
"""